# 3 STATISTICAL ANALYSIS

**Project:** Irish Company Survival: Registrations, Cohort Survival and Prediction

**Author:** Temuulen Bulgan

### Purpose of this notebook
This notebook shows how many companies were registered each year, how many were still registered 1, 3 and 5 years later, which sectors survived best, and whether Dublin companies survive longer than companies in other counties. It adds confidence intervals to the survival rates, tests whether the sector mix explains the Dublin gap, and checks whether the results depend on the definition of "closed" and the start year.

### Input
Tables in the SQL Server database `irish_company_survival` (loaded by notebook 2).

### Output
- Figures in `figures/` (`03_*.png`)
- Summary of findings and limitations at the end of this notebook

## Setup

In [2]:
import pandas as pd
from sqlalchemy import create_engine, text

#### Connect to the database

In [6]:
# Connection to the local SQL Server database
connection_string = (
    "mssql+pyodbc://@localhost/irish_company_survival"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&trusted_connection=yes"
    "&TrustServerCertificate=yes"
)
engine = create_engine(connection_string)

# Check the connection
with engine.connect() as conn:
    database_name = conn.execute(text("SELECT DB_NAME()")).scalar()

print("Connected to:", database_name)

Connected to: irish_company_survival


## 1. Load the data

I read one row per company from SQL. The query joins the fact table to the county, sector and status tables, so each company has its county, sector section and both closed flags. I use LEFT JOIN for county and sector so that companies without them are kept. Then I check the size, the missing values and the date range against notebook 2.

In [7]:
# One row per company, with county, sector and closed flags from the dimension tables
query = """
SELECT f.company_num,
       f.reg_date,
       f.closed_date,
       c.county,
       s.nace_section,
       s.section_name,
       st.closed,
       st.closed_strict
FROM fact_company AS f
LEFT JOIN dim_county AS c ON f.county_id = c.county_id
LEFT JOIN dim_sector AS s ON f.sector_id = s.sector_id
JOIN dim_status AS st ON f.status_id = st.status_id
"""
companies = pd.read_sql(text(query), engine, parse_dates=["reg_date", "closed_date"])

# Checks
print("Shape:", companies.shape, "\n")
print("Missing values:")
print(companies.isna().sum(), "\n")
print("Registration dates:", companies["reg_date"].min().date(), "to", companies["reg_date"].max().date())
print("Closing dates:     ", companies["closed_date"].min().date(), "to", companies["closed_date"].max().date())

Shape: (512008, 8) 

Missing values:
company_num           0
reg_date              0
closed_date      305953
county             1822
nace_section     246119
section_name     246119
closed                0
closed_strict         0
dtype: int64 

Registration dates: 2000-01-01 to 2026-10-27
Closing dates:      2000-07-13 to 2026-10-21


The table has 512,008 companies and 8 columns, the same as `fact_company` in notebook 2. 305,953 companies have no closing date, 1,822 have no county and 246,119 have no sector section. `nace_section` and `section_name` have the same number of missing values, so every company with a sector code has a section. Registration dates run from 2000-01-01 to 2026-10-27 and closing dates from 2000-07-13 to 2026-10-21.

## 2. Registrations per year

This section looks at how many companies were registered each year. I count the companies by registration year and check the totals against SQL Query 1.1.

In [8]:
# Year of registration, used for the yearly counts and later for the cohorts
companies["reg_year"] = companies["reg_date"].dt.year

# Number of companies registered each year
registrations = companies.groupby("reg_year").size()

# Checks against SQL Query 1.1
print("Years:", len(registrations))
print("Total:", registrations.sum(), "\n")
print(registrations)

Years: 27
Total: 512008 

reg_year
2000    18247
2001    14374
2002    13860
2003    14384
2004    15696
2005    17325
2006    19328
2007    18820
2008    14834
2009    13465
2010    14160
2011    14765
2012    14169
2013    15690
2014    17987
2015    19561
2016    21047
2017    22408
2018    22518
2019    22796
2020    21924
2021    25669
2022    21637
2023    22581
2024    23835
2025    26533
2026    24395
dtype: int64


There are 27 registration years, from 2000 to 2026, with 512,008 companies in total, the same as SQL Query 1.1. 2026 is not a full year: the data ends on 2026-10-27 (section 1), so its count of 24,395 covers less than ten months.